# Problem Book P1, Chapter 4: Symmetry Counts (Python companion)

Part A checks every number of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `sympy`, `scipy` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import itertools, math, random
from math import comb, factorial, gcd
from fractions import Fraction as Fr
from collections import Counter
import numpy as np
import sympy as sp
from sympy import totient, mobius, divisors, divisor_sigma
from scipy import integrate, special
import matplotlib.pyplot as plt
random.seed(4)

def compose(a, b):            # (a b)(i) = a(b(i)): first b, then a, as in Tome 2
    return tuple(a[i] for i in b)
def inverse(a):
    r = [0] * len(a)
    for i, v in enumerate(a): r[v] = i
    return tuple(r)
def cycle_type(g):
    seen, t = set(), Counter()
    for a in range(len(g)):
        if a in seen: continue
        L, b = 0, a
        while b not in seen: seen.add(b); b = g[b]; L += 1
        t[L] += 1
    return t
def z(t):                     # n!/z is the size of the class of type t
    return math.prod(j**m * factorial(m) for j, m in t.items())
def types(n):
    for p in sp.utilities.iterables.partitions(n):
        yield Counter(dict(p))
def egf(weights, N):          # coefficients c_n of exp(sum_j weights[j] x^j), exactly
    c = [Fr(1)] + [Fr(0)] * N
    for n in range(1, N + 1):
        c[n] = sum(j * weights.get(j, 0) * c[n - j] for j in range(1, n + 1)) / n
    return c

## Part A: checks

**Problem 4.1 (dice).** The rotation group of the cube as permutations of the faces $U,D,F,B,L,R$, then the orbits on
numberings.

In [ ]:
a = (0, 1, 5, 4, 2, 3)   # quarter turn about the vertical axis: F->R->B->L->F
b = (2, 3, 1, 0, 4, 5)   # quarter turn about the left-right axis: U->F->D->B->U
G = {tuple(range(6))}; todo = [tuple(range(6))]
while todo:
    g = todo.pop()
    for h in (a, b):
        k = compose(h, g)
        if k not in G: G.add(k); todo.append(k)
G = list(G); print(len(G))
def act(g, f): return tuple(f[inverse(g)[i]] for i in range(6))   # (g.f)(face) = f(g^-1 face)
def orbits(X):
    seen, c = set(), 0
    for x in X:
        if x not in seen: c += 1; seen |= {act(g, x) for g in G}
    return c
labs = list(itertools.permutations(range(1, 7)))
std = [l for l in labs if l[0] + l[1] == l[2] + l[3] == l[4] + l[5] == 7]
print(orbits(labs), len(std), orbits(std))

**Problem 4.2 (patterns).** Colourings up to rotation *and* renaming of the colours, by brute force against the
formulas: binary necklaces up to colour exchange, and cube faces with at most $k$ colours.

In [ ]:
def neck_brute(n):
    seen, c = set(), 0
    for x in itertools.product((0, 1), repeat=n):
        if x in seen: continue
        c += 1
        for r in range(n):
            y = x[r:] + x[:r]; seen.add(y); seen.add(tuple(1 - v for v in y))
    return c
def neck_formula(n):
    s = sum(2**gcd(m, n) for m in range(n)) + sum(2**gcd(m, n) for m in range(n) if (n // gcd(m, n)) % 2 == 0)
    return Fr(s, 2 * n)
print([(n, neck_brute(n), neck_formula(n)) for n in range(1, 13)])
def cube_patterns(k):
    seen, c = set(), 0
    for x in itertools.product(range(k), repeat=6):
        if x in seen: continue
        c += 1
        for g in G:
            y = act(g, x)
            for s in itertools.permutations(range(k)): seen.add(tuple(s[v] for v in y))
    return c
print([cube_patterns(k) for k in (2, 3, 4)])

**Problem 4.3 (necklaces).** $M(n,a)$ by brute force, Gauss's congruence for negative $a$, the cyclotomic identity up to
$x^{14}$, and the Metropolis--Rota identity.

In [ ]:
def M(n, a): return sum(mobius(d) * a**(n // d) for d in divisors(n)) // n
def prim_necklaces(n, a):
    seen, c = set(), 0
    for w in itertools.product(range(a), repeat=n):
        if w in seen: continue
        rots = {w[i:] + w[:i] for i in range(n)}; seen |= rots; c += (len(rots) == n)
    return c
print(all(M(n, a) == prim_necklaces(n, a) for n in range(1, 9) for a in (2, 3) if a**n < 10**4))
print(all(sum(mobius(d) * a**(n // d) for d in divisors(n)) % n == 0 for n in range(1, 40) for a in range(-12, 13)))
N = 15
def mul(u, v): return [sum(u[i] * v[k - i] for i in range(k + 1)) for k in range(N)]
for a in (2, 3, 5):
    r = [1] + [0] * (N - 1)
    for n in range(1, N):
        f = [0] * N
        for m in range(0, N, n): f[m] = comb(M(n, a) + m // n - 1, m // n)   # (1-x^n)^(-M)
        r = mul(r, f)
    print(a, r == [a**k for k in range(N)])
lcm = lambda i, j: i * j // gcd(i, j)
print(all(M(n, a * b) == sum(gcd(i, j) * M(i, a) * M(j, b) for i in divisors(n) for j in divisors(n) if lcm(i, j) == n)
          for n in range(1, 25) for a in (1, 2, 3) for b in (2, 5)))

**Problem 4.4 (unlabelled graphs).** Burnside over cycle types gives $g_n$; then $n!g_n/2^N$ against the two-term formula.

In [ ]:
def pair_cycles(t):
    parts = [j for j, m in t.items() for _ in range(m)]
    return sum(j // 2 for j in parts) + sum(gcd(parts[i], parts[k]) for i in range(len(parts)) for k in range(i + 1, len(parts)))
def g(n): return sum(Fr(2**pair_cycles(t), z(t)) for t in types(n))
print([int(g(n)) for n in range(1, 12)])
for n in (6, 8, 10, 14, 18, 24, 30):
    r = float(g(n) * factorial(n) / 2**comb(n, 2))
    print(n, r, 1 + n * (n - 1) / 2**(n - 1), (r - 1 - n * (n - 1) / 2**(n - 1)) * 2**(1.5 * n) / n**4)

The last column is bounded (it even tends to $0$, since the true next term is smaller than our bound), as part (c) claims.

**Problem 4.5 (commuting tuples).** Brute-force counts of commuting pairs and triples for $n\le6$ against the two products,
and the subgroup counts $a_m(\Z^3)=\sum_{e\mid m}e\,\sigma(e)$ against Hermite normal forms.

In [ ]:
def commuting_counts(n):
    Sn = list(itertools.permutations(range(n)))
    comm = {x: [y for y in Sn if compose(x, y) == compose(y, x)] for x in Sn}
    pairs = sum(len(v) for v in comm.values())
    triples = sum(sum(1 for w in comm[y] if w in set(comm[x])) for x in Sn for y in comm[x])
    return pairs, triples
def product_series(exponent, N):      # prod_j (1-x^j)^(-exponent(j)) up to x^(N-1)
    r = [1] + [0] * (N - 1)
    for j in range(1, N):
        f = [0] * N
        for m in range(0, N, j): f[m] = comb(exponent(j) + m // j - 1, m // j)
        r = [sum(r[i] * f[k - i] for i in range(k + 1)) for k in range(N)]
    return r
p2 = product_series(lambda j: 1, 9); p3 = product_series(lambda j: int(divisor_sigma(j)), 9)
for n in range(1, 7):
    h2, h3 = commuting_counts(n)
    print(n, h2 // factorial(n), p2[n], Fr(h3, factorial(n)), p3[n])
def a3_hnf(m):   # upper triangular Hermite forms diag(d1,d2,d3), d1 d2 d3 = m: d2 * d3^2 choices above the diagonal
    return sum(d2 * (m // d1 // d2)**2 for d1 in divisors(m) for d2 in divisors(m // d1))
print(all(a3_hnf(m) == sum(e * divisor_sigma(e) for e in divisors(m)) for m in range(1, 300)))

**Problem 4.6 (Landau).** Egyptian fractions $1=\sum_{i\le k}1/c_i$ with $k\le4$ terms, and the class numbers of small groups.

In [ ]:
def egyptian(k, rest=Fr(1), lo=1):
    if k == 0:
        if rest == 0: yield ()
        return
    if rest <= 0: return
    c = max(lo, math.ceil(1 / rest))
    while Fr(k, c) >= rest:
        for t in egyptian(k - 1, rest - Fr(1, c), c): yield (c,) + t
        c += 1
for k in (2, 3, 4):
    sols = list(egyptian(k)); print(k, sols, "max", max(s[-1] for s in sols), "bound", k**(2**(k - 1)))
from sympy.combinatorics.named_groups import SymmetricGroup, AlternatingGroup, DihedralGroup, CyclicGroup
def k_classes(Gp):
    els = list(Gp.generate()); seen, k = set(), 0
    for x in els:
        if x in seen: continue
        k += 1; seen |= {h * x * h**-1 for h in els}
    return k
print([(name, Gp.order(), k_classes(Gp)) for name, Gp in [("C3", CyclicGroup(3)), ("S3", SymmetricGroup(3)),
       ("C4", CyclicGroup(4)), ("D4", DihedralGroup(4)), ("A4", AlternatingGroup(4)), ("D5", DihedralGroup(5))]])

**Problems 4.7 and 4.8 (rank, derangements).** Mean and second moment of fixed points, rank, and proportion of derangements
for several transitive groups. The affine groups $t\mapsto at+b$ attain $\frac1n$.

In [ ]:
from sympy.combinatorics import Permutation, PermutationGroup
def agl(p):
    return PermutationGroup([Permutation([(i + 1) % p for i in range(p)]), Permutation([(sp.primitive_root(p) * i) % p for i in range(p)])])
def stats(Gp):
    els = list(Gp.generate()); n = Gp.degree
    F = [sum(1 for i in range(n) if x(i) == i) for x in els]
    seen, rank = set(), 0
    for pr in itertools.product(range(n), repeat=2):
        if pr in seen: continue
        rank += 1; seen |= {(x(pr[0]), x(pr[1])) for x in els}
    return Fr(sum(F), len(els)), Fr(sum(f * f for f in F), len(els)), rank, Fr(F.count(0), len(els)), Fr(1, n)
for name, Gp in [("S5", SymmetricGroup(5)), ("A5", AlternatingGroup(5)), ("D7", DihedralGroup(7)), ("D8", DihedralGroup(8)),
                 ("C6", CyclicGroup(6)), ("AGL(1,5)", agl(5)), ("AGL(1,7)", agl(7))]:
    print(name, stats(Gp))

**Problem 4.9 (fixed vectors).** Exact moments of $q^d$, $d=\dim\ker(A-I)$, over $GL_n(\mathbb F_q)$, and the probability
of the eigenvalue $1$ between $\frac1q$ and $\frac1{q-1}$.

In [ ]:
def rank_mod(rows, q):
    Mx = [list(r) for r in rows]; r = 0
    for c in range(len(Mx[0])):
        piv = next((i for i in range(r, len(Mx)) if Mx[i][c] % q), None)
        if piv is None: continue
        Mx[r], Mx[piv] = Mx[piv], Mx[r]; inv = pow(Mx[r][c], -1, q); Mx[r] = [v * inv % q for v in Mx[r]]
        for i in range(len(Mx)):
            if i != r and Mx[i][c] % q:
                f = Mx[i][c]; Mx[i] = [(x - f * y) % q for x, y in zip(Mx[i], Mx[r])]
        r += 1
    return r
for n, q in [(2, 2), (2, 3), (3, 2), (2, 5), (3, 3)]:
    tot = s1 = s2 = fix = 0
    for e in itertools.product(range(q), repeat=n * n):
        A = [e[i * n:(i + 1) * n] for i in range(n)]
        if rank_mod(A, q) < n: continue
        d = n - rank_mod([[(A[i][j] - (i == j)) % q for j in range(n)] for i in range(n)], q)
        tot += 1; s1 += q**d; s2 += q**(2 * d); fix += d > 0
    print(n, q, tot, Fr(s1, tot), Fr(s2, tot), "P(eigenvalue 1) =", round(fix / tot, 4), "bounds", round(1 / q, 4), round(1 / (q - 1), 4))

**Problem 4.10 ($k$-subsets).** The inner products $\langle\pi_k,\pi_l\rangle$ in $\mathfrak S_7$, and the degrees
$\binom nk-\binom n{k-1}$.

In [ ]:
n = 7
def pi_k(t, k):   # fixed k-subsets of a permutation of type t: choose whole cycles of total length k
    parts = [j for j, m in t.items() for _ in range(m)]
    return sum(1 for S in itertools.product((0, 1), repeat=len(parts)) if sum(p for p, s in zip(parts, S) if s) == k)
T = list(types(n))
ip = lambda k, l: sum(Fr(pi_k(t, k) * pi_k(t, l), z(t)) for t in T)
print([[ip(k, l) for l in range(4)] for k in range(4)])
print([comb(n, k) - (comb(n, k - 1) if k else 0) for k in range(4)])

**Problems 4.11 and 4.12 (cycles of a random permutation, records).** Exact probabilities in $\mathfrak S_7$, and Foata's
transformation checked as a bijection that sends cycles to records.

In [ ]:
n = 7; Sn = list(itertools.permutations(range(n)))
def cycle_of(g, a):
    s, b = {a}, g[a]
    while b != a: s.add(b); b = g[b]
    return s
print(Counter(len(cycle_of(g, 0)) for g in Sn))
print([Fr(sum(1 for g in Sn if set(range(k)) <= cycle_of(g, 0)), len(Sn)) for k in (1, 2, 3, 4, 5)])
def foata(g):
    cycles, seen = [], set()
    for a in range(len(g)):
        if a in seen: continue
        c = sorted(cycle_of(g, a)); m = max(c); cyc = [m]; b = g[m]
        while b != m: cyc.append(b); b = g[b]
        seen |= set(c); cycles.append(cyc)
    cycles.sort(key=lambda c: c[0])
    return tuple(x for c in cycles for x in c)
def records(w): return sum(1 for i in range(len(w)) if all(w[i] > w[j] for j in range(i)))
W = [foata(g) for g in Sn]
print(len(set(W)) == len(Sn), all(records(w) == sum(cycle_type(g).values()) for g, w in zip(Sn, W)))
print(foata((2, 3, 0, 1, 4)), "for (2 0)(3 1)(4) in 0-based notation")

**Problem 4.13 ($\sigma^k=e$).** The exponential generating function, Frobenius's divisibility and the congruence modulo
$p(p-1)$.

In [ ]:
for k in (2, 3, 5, 6):
    c = egf({d: Fr(1, d) for d in divisors(k)}, 40)
    a_n = [int(c[n] * factorial(n)) for n in range(41)]
    print(k, a_n[:10], all(a_n[n] % gcd(k, factorial(n)) == 0 for n in range(1, 41)))
for p in (2, 3, 5, 7):
    c = egf({1: 1, p: Fr(1, p)}, 40); a_n = [int(c[n] * factorial(n)) for n in range(41)]
    print(p, all(a_n[n] % (p * (p - 1)) == p % (p * (p - 1)) for n in range(p, 41)))
# brute force in S_6 for k = 3
print(sum(1 for g in itertools.permutations(range(6)) if compose(g, compose(g, g)) == tuple(range(6))))

For $p=2$ the congruence modulo $p(p-1)=2$ just says that $a_n(2)$ is even.

**Problems 4.14 and 4.15 (odd cycles, squares).** $o_n=e_n=((n-1)!!)^2$; squares by brute force; Blum's asymptotic.

In [ ]:
N = 60
co = egf({j: Fr(1, j) for j in range(1, N + 1, 2)}, N); ce = egf({j: Fr(1, j) for j in range(2, N + 1, 2)}, N)
dfact = lambda m: math.prod(range(m, 0, -2))
print(all(co[n] * factorial(n) == ce[n] * factorial(n) == dfact(n - 1)**2 for n in range(2, N + 1, 2)))
print(all(co[n] * factorial(n) == n * co[n - 1] * factorial(n - 1) for n in range(1, N + 1, 2)))
print([round(float(co[n]) * math.sqrt(math.pi * n / 2), 4) for n in (10, 30, 60)])
def square_type(t): return all(m % 2 == 0 for j, m in t.items() if j % 2 == 0)
psq = lambda n: sum(Fr(1, z(t)) for t in types(n) if square_type(t))
for n in range(1, 8):
    Sn = list(itertools.permutations(range(n)))
    assert Fr(len({compose(x, x) for x in Sn}), factorial(n)) == psq(n)
print([psq(n) for n in range(1, 9)])
print(all(psq(2 * m + 1) == psq(2 * m) for m in range(1, 15)))
# Blum's constant, and the ratio up to n = 800 with floating point series
NN = 800
odd = np.zeros(NN + 1); odd[0] = 1
for n in range(1, NN + 1): odd[n] = sum(odd[n - j] for j in range(1, n + 1, 2)) / n
s = odd.copy()
for m in range(1, NN // 2 + 1):
    f = np.zeros(NN + 1); k = 0
    while 4 * m * k <= NN: f[4 * m * k] = math.exp(-2 * k * math.log(2 * m) - math.lgamma(2 * k + 1)); k += 1
    s = np.convolve(s, f)[:NN + 1]
Cb = math.prod(math.cosh(1 / (2 * m)) for m in range(1, 10**6)); print("Blum constant", Cb)
print([(n, round(s[n] / (Cb * math.sqrt(2 / (math.pi * n))), 4)) for n in (50, 100, 200, 400, 800)])

**Problem 4.16 (Poisson cycle counts).** Exact factorial moments in $\mathfrak S_7$, the exact law of $C_1$ at $n=30$ against
Poisson$(1)$, and the probability of no short cycle.

In [ ]:
n = 7; Sn = list(itertools.permutations(range(n))); TT = [cycle_type(g) for g in Sn]
ff = lambda c, m: math.prod(c - i for i in range(m))
for ms in [(2, 1, 0), (0, 0, 2), (1, 1, 1), (3, 2, 0)]:
    E = Fr(sum(math.prod(ff(t[j + 1], m) for j, m in enumerate(ms)) for t in TT), len(Sn))
    M_ = sum((j + 1) * m for j, m in enumerate(ms))
    print(ms, E, math.prod(Fr(1, (j + 1)**m) for j, m in enumerate(ms)) if M_ <= n else 0)
def no_j(m, j): return sum(Fr((-1)**i, j**i * factorial(i)) for i in range(m // j + 1))
print(max(abs(float(Fr(1, factorial(k)) * no_j(30 - k, 1)) - math.exp(-1) / factorial(k)) for k in range(31)))
for m in (2, 3, 5, 10):
    c = egf({j: Fr(1, j) for j in range(m, 61)}, 60)
    print(m, float(c[60]), math.exp(-sum(1 / j for j in range(1, m))), math.exp(-0.5772156649) / m)

**Problem 4.17 (Golomb--Dickman).** The two integrals, and the exact $\mathbb E[L_n]/n$.

In [ ]:
E1 = special.exp1
print(integrate.quad(lambda u: 1 - math.exp(-E1(u)), 0, np.inf, limit=200)[0],
      integrate.quad(lambda u: math.exp(-u - E1(u)), 0, np.inf, limit=200)[0])
def EL(n):
    return sum(1 - egf({j: Fr(1, j) for j in range(1, m + 1)}, n)[n] for m in range(n))
print([(n, round(float(EL(n) / n), 4)) for n in (5, 10, 20, 40, 60)])
# the Poissonised identity of (b) at x = 0.9, truncated
x = 0.9
rhs = sum(1 - math.exp(-sum(x**j / j for j in range(m + 1, 2000))) for m in range(0, 1500))
t = -1 / math.log(x); print(rhs / t)

At $x=0.9$ ($t\approx9.5$) the ratio is already close to $\lambda=0.6243$; the $O(\log t)$ correction of part (c) explains
the gap.

**Problem 4.18 (conjugate permutations).** $P_n=\sum_\lambda z_\lambda^{-2}$, the infinite product, and $n^2P_n$.

In [ ]:
P = lambda n: sum(Fr(1, z(t)**2) for t in types(n))
Cc = math.prod(special.i0(2 / j) for j in range(1, 10**6)); print(Cc, float(sum(P(n) for n in range(0, 60))))
print([(n, round(float(P(n)) * n * n, 3)) for n in (10, 20, 40, 60)])

The partial sum up to $59$ is close to the product, and $n^2P_n$ comes down slowly towards it, from above.

**Problem 4.19 (commutators).** Counts in $A_5$ by brute force against Frobenius's formula, and in $\mathfrak S_3$.

In [ ]:
def even(p): return sum(p[i] > p[j] for i in range(len(p)) for j in range(i + 1, len(p))) % 2 == 0
A5 = [p for p in itertools.permutations(range(5)) if even(p)]
cnt = Counter(compose(compose(x, y), compose(inverse(x), inverse(y))) for x in A5 for y in A5)
by_type = {}
for g, c in cnt.items(): by_type.setdefault(tuple(sorted(cycle_type(g).items())), set()).add(c)
print(len(cnt), by_type)
phi = (1 + 5**.5) / 2
table = [[1, 1, 1, 1, 1], [3, -1, 0, phi, 1 - phi], [3, -1, 0, 1 - phi, phi], [4, 0, 1, -1, -1], [5, 1, -1, 0, 0]]
print([round(60 * sum(ch[i] / ch[0] for ch in table), 6) for i in range(5)])
S3 = list(itertools.permutations(range(3)))
print(Counter(compose(compose(x, y), compose(inverse(x), inverse(y))) for x in S3 for y in S3))

**Problem 4.20 (Frobenius--Schur).** The sum of the degrees of $\mathfrak S_n$ (hook length formula) against the number of
involutions, and $r(g)$ for $Q_8$.

In [ ]:
def degrees(n):
    out = []
    for p in sp.utilities.iterables.partitions(n):
        lam = sorted([j for j, m in p.items() for _ in range(m)], reverse=True)
        conj = [sum(1 for l in lam if l > i) for i in range(lam[0])]
        hooks = math.prod((l - j - 1) + (conj[j] - i - 1) + 1 for i, l in enumerate(lam) for j in range(l))
        out.append(factorial(n) // hooks)
    return out
inv = [1, 1]
for n in range(2, 16): inv.append(inv[-1] + (n - 1) * inv[-2])
print([(n, sum(degrees(n)), inv[n]) for n in range(1, 11)], all(sum(degrees(n)) == inv[n] for n in range(1, 16)))
# Q8 as unit quaternions {±1, ±i, ±j, ±k}
import numpy.linalg as la
one = np.eye(2); I = np.array([[1j, 0], [0, -1j]]); J = np.array([[0, 1], [-1, 0]]); K = I @ J
Q8 = [s * m for s in (1, -1) for m in (one, I, J, K)]
print("solutions of x^2 = 1 in Q8:", sum(1 for x in Q8 if np.allclose(x @ x, one)))

**Problem 4.21 (products of two $n$-cycles).** Brute force for $n\le7$, the eigenvalue identity, and the reciprocal-binomial sum.

In [ ]:
for n in range(3, 8):
    Sn = list(itertools.permutations(range(n)))
    cyc = [g for g in Sn if cycle_type(g) == Counter({n: 1})]; S = set(cyc)
    c = tuple((i + 1) % n for i in range(n))
    print(n, sum(1 for a_ in cyc if compose(inverse(a_), c) in S), Fr(2 * factorial(n - 1), n + 1) if n % 2 else 0)
print(all(sum(Fr((-1)**r, comb(m, r)) for r in range(m + 1)) == Fr(m + 1, m + 2) * (1 + (-1)**m) for m in range(0, 40)))
for n in (5, 6):
    zeta = np.exp(2j * np.pi / n)
    print(n, np.round(np.poly([-zeta**j for j in range(1, n)])[::-1].real, 8))   # coefficients of prod (t + zeta^j) ~ e_r

The coefficients of $\prod_{j=1}^{n-1}(t+\zeta^j)$ are $e_{n-1-r}$; they are all $\pm1$, alternating, as part (a) says.

**Problem 4.22 (Dénes).** Factorisations of the $n$-cycle into $n-1$ transpositions, and the tree test on all of them.

In [ ]:
def is_tree(edges, n):
    parent = list(range(n))
    def find(a):
        while parent[a] != a: a = parent[a]
        return a
    for a_, b_ in edges:
        ra, rb = find(a_), find(b_)
        if ra == rb: return False
        parent[ra] = rb
    return True
for n in (3, 4, 5, 6):
    target = tuple((i + 1) % n for i in range(n))
    Tr = list(itertools.combinations(range(n), 2)); count = 0; ok = True
    for seq in itertools.product(Tr, repeat=n - 1):
        p = tuple(range(n))
        for (a_, b_) in seq:
            t = list(range(n)); t[a_], t[b_] = b_, a_; p = compose(p, tuple(t))
        is_ncyc = cycle_type(p) == Counter({n: 1})
        ok &= is_ncyc == is_tree(seq, n)
        count += p == target
    print(n, count, n**(n - 2), ok)

## Part B: laboratory

**L4.1 (the Chinese restaurant).** Build a uniform random permutation of $\{1,\dots,n\}$ one point at a time: point $k$ either
opens a new cycle (probability $\frac1k$) or is inserted after a uniformly random earlier point. Check that the result is
uniform for $n=4$, then simulate $n=10^5$ many times and compare the longest cycle with $\lambda n$ and the cycle counts with
Poisson laws.

**L4.2 (derangements in permutation groups).** For the transitive groups $\mathfrak S_n$, $A_n$, $D_n$, the affine groups
$t\mapsto at+b$ of $\mathbb F_p$, and $PGL_2(\mathbb F_q)$ acting on the $q+1$ points of the projective line, compute the
proportion of derangements and compare with $\frac1n$. Which groups come close to the bound?

**L4.3 (commuting quadruples).** Use the recursion $a_m(\Z^k)=\sum_{e\mid m}e\,a_e(\Z^{k-1})$ to compute
$\sum_nh_4(n)\frac{x^n}{n!}$ up to $x^{10}$, check $h_4(n)$ by brute force for $n\le4$, and check the Dirichlet series identity
$\sum_ma_m(\Z^3)m^{-s}=\zeta(s)\zeta(s-1)\zeta(s-2)$ numerically at $s=4$.

**L4.4 (asymmetric graphs).** Sample uniform random graphs on $n=6,7,8$ vertices and test whether their automorphism group is
trivial. Compare the proportion of symmetric graphs with $\frac{n(n-1)}{2^{n-1}}$, the transposition term of Problem 4.4.

## Solutions of the laboratory

### L4.1

In [ ]:
def restaurant(n, rng=random):
    nxt = []                       # nxt[i] = image of i
    for k in range(n):
        if rng.random() < 1 / (k + 1):
            nxt.append(k)          # new fixed point
        else:
            j = rng.randrange(k)   # insert k after j in j's cycle
            nxt.append(nxt[j]); nxt[j] = k
    return tuple(nxt)
cnt = Counter(restaurant(4) for _ in range(48000))
print(len(cnt), min(cnt.values()), max(cnt.values()), "expected about", 48000 // 24)
n, R = 10**5, 300
Ls, C1, C2 = [], [], []
for _ in range(R):
    g = restaurant(n); t = cycle_type(g)
    Ls.append(max(t) / n); C1.append(t[1]); C2.append(t[2])
print("mean L/n", np.mean(Ls), "lambda", 0.62433)
print("C1 law", [round(C1.count(k) / R, 3) for k in range(4)], [round(math.exp(-1) / factorial(k), 3) for k in range(4)])
print("C2 law", [round(C2.count(k) / R, 3) for k in range(4)], [round(math.exp(-0.5) * 0.5**k / factorial(k), 3) for k in range(4)])
plt.figure(figsize=(6, 3)); plt.hist(Ls, bins=30, density=True); plt.xlabel("L/n"); plt.show()

The restaurant gives the uniform law (each of the $24$ permutations about $2000$ times), and the simulated longest cycle
averages close to $0.62n$. The histogram of $L/n$ changes shape at $\frac12$: on $[\frac12,1]$ the limiting density is exactly
$\frac1x$ (its integral is the $\log2$ of Tome~3, Exercise~2.6), and below $\frac12$ it bends down, because a longest cycle
shorter than $\frac n2$ leaves room for a second long one.

### L4.2

In [ ]:
def pgl2(q):
    pts = [(1, b) for b in range(q)] + [(0, 1)]            # (x, y) projective, normalised
    def norm(v):
        x_, y_ = v[0] % q, v[1] % q
        if x_: inv_ = pow(x_, -1, q); return (1, y_ * inv_ % q)
        return (0, 1)
    idx = {p: i for i, p in enumerate(pts)}
    gens = []
    for M_ in ([[1, 1], [0, 1]], [[sp.primitive_root(q), 0], [0, 1]], [[0, 1], [1, 0]]):
        gens.append(Permutation([idx[norm((M_[0][0] * x_ + M_[0][1] * y_, M_[1][0] * x_ + M_[1][1] * y_))] for (x_, y_) in pts]))
    return PermutationGroup(gens)
def der(Gp):
    els = list(Gp.generate()); n = Gp.degree
    return Fr(sum(1 for x in els if all(x(i) != i for i in range(n))), len(els)), Fr(1, n)
for name, Gp in [("S6", SymmetricGroup(6)), ("A6", AlternatingGroup(6)), ("D9", DihedralGroup(9)), ("AGL(1,11)", agl(11)),
                 ("PGL2(5)", pgl2(5)), ("PGL2(7)", pgl2(7)), ("C8", CyclicGroup(8))]:
    d, b = der(Gp); print(name, Gp.order(), d, float(d), "bound", b)

The affine groups sit exactly on the bound, as Problem 4.8 predicts for sharply $2$-transitive groups. $PGL_2(\mathbb F_q)$ is
sharply $3$-transitive, and its derangement proportion is $\frac q{2(q+1)}$ ($\frac5{12}$ and $\frac7{16}$ here), close to $\frac12$
and far above $\frac1{q+1}$: an element $\ne e$ fixes $0$, $1$ or $2$ points, and the elements with no fixed point are those whose
characteristic polynomial is irreducible. Symmetric and alternating groups give about $e^{-1}$, cyclic groups $1-\frac1n$.

### L4.3

In [ ]:
def a_sub(m, k):
    if k == 1: return 1
    return sum(e * a_sub(e, k - 1) for e in divisors(m))
N = 11
log_coeffs = {m: Fr(a_sub(m, 4), m) for m in range(1, N)}
c4 = egf(log_coeffs, N - 1)
print([int(c4[n] * factorial(n)) for n in range(N)])
def h4_brute(n):
    Sn = list(itertools.permutations(range(n)))
    comm = {x: set(y for y in Sn if compose(x, y) == compose(y, x)) for x in Sn}
    return sum(len(comm[x] & comm[y] & comm[w]) for x in Sn for y in comm[x] for w in comm[x] & comm[y])
print([h4_brute(n) for n in range(5)])
s = 4
lhs = sum(a_sub(m, 3) / m**s for m in range(1, 4000))
print(lhs, float(sp.zeta(s) * sp.zeta(s - 1) * sp.zeta(s - 2)))

The brute-force counts $1,1,16,126,2016$ agree with $n!\,[x^n]$ of the exponential, and the Dirichlet series matches
$\zeta(4)\zeta(3)\zeta(2)$ up to the truncation error (the terms decay like $m^{-2}\log m$ at $s=4$).

### L4.4

In [ ]:
rng = np.random.default_rng(4)
for n in (6, 7, 8):
    P_ = np.array(list(itertools.permutations(range(n))))[1:]       # non-identity permutations
    sym, R = 0, 400 if n < 8 else 150
    for _ in range(R):
        U = np.triu(rng.integers(0, 2, (n, n)), 1); A = U + U.T
        B = A[P_[:, :, None], P_[:, None, :]]                       # A permuted by each p
        sym += bool(np.any(np.all(B == A, axis=(1, 2))))
    print(n, "symmetric proportion", sym / R, "transposition term", n * (n - 1) / 2**(n - 1))

At these sizes many graphs still have a symmetry, most often the transposition of two "twin" vertices with the same
other neighbours. Each pair of vertices is a twin pair with probability $2^{2-n}$, so the transposition term $\binom n22^{2-n}$
is the expected number of twin pairs, an upper bound for their probability that becomes sharp when it is small. The decay to
$0$ is exponential but starts late: at $n=8$ about one graph in three is still symmetric, and the transposition term falls
below $1\%$ only at $n=16$.